# Heart disease classifier tuning

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/mahatiratusher/heart-disease-risk-prediction-dataset/heart_disease_risk_dataset_earlymed.csv')
df.head()

## Basic checks

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df['Heart_Risk'].value_counts()

## Cleaning

In [ ]:
# Remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

df.shape

## Split the data

In [ ]:
X = df.drop('Heart_Risk', axis=1)
y = df['Heart_Risk']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Baseline Decision Tree

In [ ]:
baseline_model = DecisionTreeClassifier(random_state=42)

baseline_cv_scores = cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

baseline_cv_accuracy = baseline_cv_scores.mean()
baseline_test_accuracy = accuracy_score(y_test, baseline_pred)

print('CV Accuracy:', baseline_cv_accuracy)
print('Test Accuracy:', baseline_test_accuracy)

## GridSearchCV

In [ ]:
grid_params = {
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 4]
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    grid_params,
    cv=5,
    scoring='accuracy',
    n_jobs=1
)

grid_search.fit(X_train, y_train)

grid_pred = grid_search.predict(X_test)
grid_cv_accuracy = grid_search.best_score_
grid_test_accuracy = accuracy_score(y_test, grid_pred)

print('Best Parameters:', grid_search.best_params_)
print('CV Accuracy:', grid_cv_accuracy)
print('Test Accuracy:', grid_test_accuracy)

## RandomizedSearchCV

In [ ]:
random_params = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 15, 20],
    'min_samples_leaf': [1, 2, 4, 6, 8]
}

random_search = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=42),
    random_params,
    n_iter=10,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=1
)

random_search.fit(X_train, y_train)

random_pred = random_search.predict(X_test)
random_cv_accuracy = random_search.best_score_
random_test_accuracy = accuracy_score(y_test, random_pred)

print('Best Parameters:', random_search.best_params_)
print('CV Accuracy:', random_cv_accuracy)
print('Test Accuracy:', random_test_accuracy)

## Random Forest ensemble

In [ ]:
forest_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=15,
    random_state=42,
    n_jobs=1
)

forest_cv_scores = cross_val_score(
    forest_model,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy',
    n_jobs=1
)

forest_model.fit(X_train, y_train)
forest_pred = forest_model.predict(X_test)

forest_cv_accuracy = forest_cv_scores.mean()
forest_test_accuracy = accuracy_score(y_test, forest_pred)

print('CV Accuracy:', forest_cv_accuracy)
print('Test Accuracy:', forest_test_accuracy)

## Results

In [ ]:
results = pd.DataFrame({
    'Model': [
        'Baseline Decision Tree',
        'GridSearchCV Decision Tree',
        'RandomizedSearchCV Decision Tree',
        'Random Forest'
    ],
    'CV Accuracy': [
        baseline_cv_accuracy,
        grid_cv_accuracy,
        random_cv_accuracy,
        forest_cv_accuracy
    ],
    'Test Accuracy': [
        baseline_test_accuracy,
        grid_test_accuracy,
        random_test_accuracy,
        forest_test_accuracy
    ],
    'Best Params': [
        'Default parameters',
        str(grid_search.best_params_),
        str(random_search.best_params_),
        "{'n_estimators': 50, 'max_depth': 15}"
    ]
})

results

## Conclusion

In [ ]:
best_result = results.loc[results['Test Accuracy'].idxmax()]

print(f"{best_result['Model']} performed best with a test accuracy of {best_result['Test Accuracy']:.4f}.")
print("GridSearchCV checks every parameter combination, while RandomizedSearchCV tests a smaller random sample and usually finishes faster.")
print("The Random Forest combines many decision trees, which usually gives more stable predictions than one tree.")